In [3]:
import numpy as np
import sys
import scipy

from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib import cm

# Neutron Diffusion in a Bare Spherical Fissile Assembly

**M6023 Advanced Quantitative Modelling: Assessment 3**

**Student ID:** 23360932014

This notebook develops a one-group neutron diffusion model for a bare spherical fissile assembly.

## Contents

1. Background


---

## 1. Background

### Why does critical mass exist?
In July 1945, in the New Mexico desert, a sphere of plutonium that was roughly the size of a grapefruit was utterly compressed by a precisely timed arrangement of conventional explosives. Within microseconds, the compression pushed the plutonium past a particular threshold: the neutrons inside it stopped leaking away faster than they were being produced, and instead the population of neutrons began to grow. Each fission event released neutrons that struck other nuclei and caused further fissions, which released even more neutrons, and so on. The resulting chain reaction released the energy of roughly twenty thousand tonnes of TNT.

We have a name for what was just described: **criticality**. And the quantity that determined whether the sphere was above or below the threshold of criticality is one of the most consequential numbers in twentieth-century physics: known as the **critical mass**.

Below this threshold, a lump of fissile material is simply inert. A few neutrons may wander here and there through it, causing a few fissions, but most of the neutrons subsequently created escape out of the surface before they can find another nucleus to split. Above the threshold, the same material sustains a chain reaction. The difference between "inert lump" and "exponentially growing neutron population" is actually a matter of geometry: a sphere of radius $R$ just slightly larger than some critical value $R_c$ is qualitatively different from one just slightly smaller, even though the material itself is identical.

So, the question this notebook will attempt to answer is: **given a fissile material with known microscopic properties, what is $R_c$?**

### What is the physical picture?

Right, so why does this threshold exist at all? To understand that, we need to think about what actually happens to a neutron inside a lump of fissile material. It can meet one of three fates:

1. It gets **absorbed** by a nucleus without causing a fission at all. This is like a dud interaction, as far as the chain reaction is concerned. The neutron is just gone to the "wind".

2. It causes a **fission**, in which case the nucleus splits and releases on average $\nu$ new neutrons (typically between 2.4 and 2.9, depending on our isotope). This is the productive outcome that is typically searched for in industrial reactions. 

3. It travels all the way to the **surface** of the material and escapes into the surrounding vacuum, lost forever and forgotten.

Whether the neutron population grows, holds steady, or dies out really depends entirely on the balance between these three fates. Fission, as a process, creates neutrons at a rate proportional to how many are already present and how likely each one is to cause a fission. Absorption removes them at a rate proportional to how many are present and how likely each is to be absorbed. Pretty straightforward, right?

Maybe not. You see, leakage (neutrons escaping) works differently. It depends not on the total number of neutrons but actually on how they are distributed through the material. Neutrons only leak from regions near the surface, because a neutron deep in the interior is unlikely to make it to the outside without first bumping into something (probably another neutron). 

So, what does this mean? Well, it means leakage scales roughly with **surface area**, while production and absorption scale with **volume**.

As such, when a sphere gets larger, volume grows as $R^3$ but surface area only grows as $R^2$. So the *fraction* of neutrons that can escape decreases as the sphere gets bigger. Below a certain size, leakage just dominates and the population of neutrons dies out. Above it, production dominates and the population grows. At exactly $R_c$, the two balance. 

This is why criticality depends on geometry as much as on material itself.

### Why is this a PDE problem?

So, we've established that criticality depends on a balance in the third demnsion: neutrons near the surface leak out, neutrons in the interior cause fissions. 

But how do we actually turn that into mathematics? And why do we care? 

Well, we need a quantity that captures how neutrons are distributed across space and time. This is the **neutron flux** $\varphi(\mathbf{r}, t)$, which has units of neutrons per square centimetre per second and measures. Or, in looser terms, how many neutrons are passing through a given point per unit time. By the way, this is a field, meaning it will take a value at every point in space and every instant in time.

The three physical processes that we talked about earlier correspond to three mathematical terms governing how $\varphi$ evolves:

- **Production and absorption** act locally, at each point in our field, and depend on the flux (neutron "traffic") at that point.
- **Leakage** acts through gradients in space: neutrons tend to flow from regions of high flux to regions of low flux, like how heat flows from hot regions to cold. This is our **diffusion** mechanism, and we describe it mathematically as the Laplacian operator $\nabla^2$.

This Laplacian is a spatial second derivative. What it does is that it measures how much a quantity at a point differs from the average of its neighbours. If you've done the heat equatione literally ever (and god knows I have), you'll know this is the exact same mechanism. The only tangible difference here is that on top of diffusion, we also have a reactive source term: the material is producing and absorbing neutrons as well as letting them diffuse. The fission process!

So, when we put these three terms together, we get a PDE of the form:

$$\frac{\partial \varphi}{\partial t} = \underbrace{D \nabla^2 \varphi}_{\text{leakage (diffusion)}} + \underbrace{(\nu \Sigma_f - \Sigma_a)\varphi}_{\text{net production}}$$

wherein $D$ is the diffusion coefficient, $\Sigma_f$ and $\Sigma_a$ are the fission and absorption cross-sections (on a macroscopic level at least), and $\nu$ is the average number of neutrons released per fission.

THIS is why we need a PDE and not just an ODE: our leakage term involves *spatial* derivatives of $\varphi$, while that time evolution involves a *time* derivative. How these two interact and go back-and-forth is what produces criticality under the umbrella of geometrics. 

If we derive this equation properly, and then solve it for a spherical geometry, we can use it to calculate $R_c$. 

Which is the subject of this notebook.

### A bare sphere? One-group theory? What are we talking about?

Alright, before we begin, we  honestly need to do a quick explanation for two major simplifications we're about to make.

The **bare sphere** (a sphere of fissile material surrounded by nothing but vacuum) is, quite literally, the simplest geometry we can use to capture the essential physics. It has a single length scale (the radius), actually has an exact formula that we can solve directly (thus we can use it to check if our numerical answers are right), and it gives us an upper bound on the critical radius for a given material. Being honest, any real-world assembly with a reflector of uranium or beryllium surrounding the fissile core would have a much *smaller* critical radius than the bare case. Also, fun fact, the bare-sphere calculation we're making also is historically rich: the earliest fission device designs basically did this calculation (well, theirs was a touch more complicated, but, who's counting)

**The one-group diffusion theory** treats all neutrons as having the same effective energy. The reason we do this is basically because otherwise, we would have to do a SEVEN-dimensional problem (three spatial, one temporal, one energy, two angular - no thank you) to just two dimensions ($r$ and $t$) under spherical symmetry. You could use multi-group diffusion or even, god forbid, the full neutron transport equation, but those wouldn't really change things for our bare sphere. So, it would just be making things more complicated, and one-group theory gives us values with 10 or 20 per cent of the experimental values. And even better, the sources of that error are well documented. 

We will discuss both of these choices in Section 2, Assumptions, and a likely section on Limitations. 

---

## 2. Modelling Assumptions

Time to run through our assumptions. And there are quite a few, so strap yourselves in. Each of these simplifications reduces the complexity of the problem (I won't complain, probably), but they also introduce limitations. We shall revisit these in the later Limitations section probably.

---
**Assumption 1: One-group energy approximation.** 

What we talked about before. Neutrons produced by fission have a *broad* energy spectrum, ranging from fast neutrons at roughly $2\,\text{MeV}$ at the time of their emission, all the way down to thermal neutrons at approximately $0.025\,\text{eV}$ after moderation (slowing down neutrons. This is what water/graphite is used for). The probability of fission, absorption, and scattering depends STRONGLY on this neutron energy, and a more thorough piece of work would solve for the flux as a function of both position *and* energy through things like the full neutron transport equation we mentioned earlier. 

What we shall do here is treat all neutrons as basically having a single effective energy, with the cross-section averaged over the spectrum of possible energies. This collapses the pesky energy dimension in its entirety. The good news is, as mentioned prior, it changes very little with our bare sphere. So, not much to worry about in terms of justification.

---
**Assumption 2: Diffusion approximation.**  

Neutron transport is modelled using Fick's law (of diffusion), which states that the neutron current $\mathbf{J}$ is proportional to the negative gradient of the flux $\varphi$:

$$\mathbf{J} = -D \nabla \varphi$$

Now, this is important: this is *not, in any way,* a fundamental law. It comes from simplifying the full neutron transport equation by keeping only the first part of its angular behaviour. In summary: the net neutron current at a point is proportional to the local gradient of the scalar flux, directed DOWN the gradient. Physically, this is basically that neutrons undergo a random walk through the material, and on average they drift from regions of high flux to low flux.

This assumption checks out when the angular distribution of neutrons is nearly isotropic, meaning that all the neutrons move equally in every possible direction, and there is no preferred paths to their trejectory. This assumption fails near boundaries (given neutrons escape but don’t come back), in materials that absorb a LOT of neutrons, and close to concentrated sources of neutron emission. 

Neutron scattering isn’t perfectly random realistically, so we correct for its directionality using $\Sigma_{\text{tr}} = \Sigma_s(1 - \bar{\mu}) + \Sigma_a$, where $\bar{\mu}$ is the mean cosine of the scattering angle. For a bare sphere of fissile material, this diffusion approximation is what gives the critical radii that are within 10 to 20 percent of experimental numbers.


---
**Assumption 3: That we have a homogeneous, isotropic medium.** 

The fissile material we are using for this work is assumed to be uniform in composition and structure throughout the sphere. There is no variation in density, isotopic composition, or cross-sections. Actual fissile cores contain structural material, voids, and all kinds of density gradients, and in some cases, they are even deliberately layered or alloyed. 

But it's this homogeneity that allows us to treat $D$, $\Sigma_f$, and $\Sigma_a$ as constants rather than functions of position within the sphere, and that is basically what makes any analytical solution possible at all. Without it, we'd need to rely on a purely numerical approach.

---
**Assumption 4: We have a bare sphere, with no reflector.** 

This assumption is one we've already covered, but it's important to emphasise it. *The fissile material is a sphere of radius $R$ surrounded by vacuum*. There is no surrounding material to reflect escaping neutrons back into the core, our sphere is in a vacuum. Floating in space perhaps (okay, not space, but it's funny to think about. Like that core from Portal 2). In actuality, most fission assemblies almost always incorporate a reflecto. These scatter leaking neutrons back into the fissile core and thus reduce the critical mass substantially, often by a factor of two or more. Our bare sphere gives us a simple, solvable estimate for the largest possible critical radius we could achieve. 


---
**Assumption 5: There are prompt neutrons only; we have no delayed neutron precursors.** 

Okay, time for a quick science lecture.

Fun fact, $99.35\%$ of neutrons produced by fission in U-235 are emitted within $10^{-14}\,\text{s}$ of the fission event. Essentially, for the purposes of our human perception of the universe, this is instantaneous. The remaining $\sim 0.65\%$ are actually emitted by other fission products with half-lives ranging from fractions of a second up to roughly a minute. 

These "delayed" neutrons are actually central to the control of real reactors because they slow the effective response time of the neutron population from microseconds to seconds, which is what makes mechanical control rods actually feasible (god bless those operators). But for the question of criticality itself (aka, "is the neutron population in a steady state or not?"), delayed neutrons can be folded into an effective $\nu$. For our later simulations, we will be treating all neutrons as prompt, so any growth and decay timescales we discuss should be understood as *prompt-neutron dynamics*, not times realistic to any actual reactor in the real-world. 



---
**Assumption 6: There is no temperature feedback.** 

Our cross-sections and diffusion coefficient are treated as constants in time. In reality, heating from fission causes expansion (basically, this reduces the density and hence the cross-sections on a macroscopic scale), and this in turn modifies the absorption in a temperature-dependent way. Over longer timescales, burnup of fissile isotopes depletes $\Sigma_f$ (the probability per distance that a neutron causes fission) while building up absorbing fission products (aka, things that slow fission). These feedback mechanisms are essential for reactor safety analysis (which I did work in at the Safety Department at Sizewell B) but they operate on timescales longer than the prompt-neutron dynamics we're modelling here in the first place. So, should probably be fine. 

---

Taken together, our six assumptions reduce the problem to a one-dimensional, time-dependent PDE in the radial coordinate $r$ (so no side-to-side or angular variation), with constant coefficients, that is linear in the flux $\varphi$, that has an exact steady-state solution we can use to check our numerical results. 

These simplifications are... severe, quite honestly, but we can quantify their consequences quantitatively thanks to well-documented research, and in general, the sphere reduces things enough anyway that the assumptions fit well with that geometry in the context!

---

## 3. Derivation of the Neutron Diffusion Equation

Right, now we actually derive the governing equation. Sit back and relax. One of us has to.

So, so our approach is the same one that is used throughout continuum physics: you take a control volume, then you write down a balance of what's coming in, what's going out, and what's being produced or destroyed inside, and finally, you shrink the volume to a single point.



### Neutron balance on a control volume

So, first, we must consider some arbitrary volume $V$ within the fissile material, bounded by a closed surface $S$. The number of neutrons in $V$ changes over time due to *three* processes (those ones I mentioned before. And trust me, we will keep doing so until the end of time): 

- Production by fission inside $V$
- Absorption by nuclei inside $V$
- Net leakage of neutrons through the surface $S$.

Now, we shall call our neutron number density (otherwise known as the neutrons per unit volume):

$$
n(\mathbf{r}, t)
$$

And, we shall our neutron current density (aka, the net number of neutrons cross a unit area per unit time in any given direction):

$$
\mathbf{J}(\mathbf{r}, t)
$$. 

Finally, the scalar neutron flux is:
$$
\varphi = n v
$$
where \( v \) is the neutron speed.




Now, to account for everything happening inside our region $V$, we form the following equation:

$$\frac{\partial}{\partial t} \int_V n \, dV = \int_V \left( \text{production} - \text{absorption} \right) dV - \oint_S \mathbf{J} \cdot d\mathbf{S}$$

Let's unpack each term on the right.



**Production** 

Every single fission event (not the same as every single atom) releases $\nu$ new neutrons. 

The rate of these fission reactions per unit volume is $\Sigma_f \varphi$, where $\Sigma_f$ is the  fission cross-section on a macroscopic level (units: cm$^{-1}$). 

This  cross-section is almost a probability. It's a measure of how *likely* a neutron is to cause a fission per unit length of its path, so multiplying it by the flux (reminder, this measures how many neutrons are passing through per unit area per unit time. Like neutron "traffic") gives us a reaction rate per unit volume. 

As such, the production rate per unit volume is:

$$\text{Production rate} = \nu \Sigma_f \varphi$$

**Absorption** 

Quite similarly, the rate of absorption reactions per unit volume is $\Sigma_a \varphi$, where $\Sigma_a$ is the macroscopic absorption cross-section. This includes *all* absorptions, both fission and non-fission captures. THIS IS IMPORTANT. 

The fission contribution is *already* counted in our production term, so we don't double dip:

$$\text{Absorption rate} = \Sigma_a \varphi$$

**Leakage** 

The surface integral $\oint_S \mathbf{J} \cdot d\mathbf{S}$ gives us the net rate of neutrons leaving $V$ through $S$. In simpler terms, how much of the neutron flow crosses every little patch of our $S$ boundary and then summing that to a total. 

Now, we can convert this to a volume integral using the divergence theorem:

$$\oint_S \mathbf{J} \cdot d\mathbf{S} = \int_V \nabla \cdot \mathbf{J} \, dV$$

This is the standard vector calculus identity that converts a surface flux into a volume integral of the divergence (the neutron flow at a single point. Basically the surface integral localised to a single point). It works for any vector field $J$ and any closed surface $S$. 

But why do we do this? Well, we want all three of our terms (that being production, absorption, and leakage) expressed as volume integrals over the same region, our $V$. This will allow us to drop our integrals and obtain the equation for what's going on at a single point. Like so:

Let's substitute all three terms back into our main equation:

$$\frac{\partial}{\partial t} \int_V n \, dV = \int_V \left( \nu \Sigma_f \varphi - \Sigma_a \varphi \right) dV - \int_V \nabla \cdot \mathbf{J} \, dV$$

Now both terms on the right are volume integrals over the same region $V$. So, we can combine them under a single integral:

$$\frac{\partial}{\partial t} \int_V n \, dV = \int_V \left( \nu \Sigma_f \varphi - \Sigma_a \varphi - \nabla \cdot \mathbf{J} \right) dV$$

Note that the minus sign on $\nabla \cdot \mathbf{J}$ is the original subtraction from our balance: leakage is still being subtracted from production and absorption. We've just moved it inside the integral.

Now, since the volume $V$ is arbitrary (we literally could have chosen any region inside the material), the integrals must be equal locally. This is actually pretty standard argumentation for localising anything in physics: if the integral of some quantity over *every possible* volume is zero, then the quantity itself must be zero everywhere else. 

As such:

$$\frac{\partial n}{\partial t} = \nu \Sigma_f \varphi - \Sigma_a \varphi - \nabla \cdot \mathbf{J}$$

The issue is now that we have unknowns. $n$ (otherwise known as the lovely $\varphi$) and $\mathbf{J}$. Now, time to bring in Fick's Law from our second assumption!

### Hello Fick's law!

So, these unknowns. What to do with them? Well, we need ANOTHER relationship to close the system. A polycule even. 

Let's bring in Fick's Law:

$$\mathbf{J} = -D \nabla \varphi$$

Remember from our assumption discussion, this is *not* a fundamental law but rather an approximation: the net neutron current at any point is proportional to the local gradient of the flux, and it points "downhill" (aka, from high flux to low flux). 

You should recognise some of that rhetoric as being pretty standard for diffusion (god bless GCSE Biology and Osmosis). Maybe not the neutron current though. Or who knows? Maybe you had a crazy good Biology teacher (or a crazy bad one if they were yapping about neutrons).


Anyway, the constant of proportionality $D$ is the diffusion coefficient, and it has units of cm (centimetres).

Now remember what I mentioned earlier about this being similar to Fourier's Law for Heat Conduction? We've got ($\mathbf{q} = -k \nabla T$) and Fick's law for chemical diffusion ($\mathbf{j} = -D \nabla c$). The physics is different, obviously, but the mathematics is the same: there are randomly-walking particles drifting down a concentration gradient.

Let's compute $\nabla \cdot \mathbf{J}$. 

First, we substitute Fick's law into the divergence from earlier:

$$\nabla \cdot \mathbf{J} = \nabla \cdot (-D \nabla \varphi)$$

Now! Since, $D$ is spatially constant (the humble homogenous material, as per Assumption 3), we've just got a simple number multiplying the vector field $\nabla \varphi$. Let's take that guy out, shall we?

$$\nabla \cdot \mathbf{J} = -D \, \nabla \cdot (\nabla \varphi)$$

Now, $\nabla \cdot (\nabla \varphi)$ is the divergence of the gradient of $\varphi$. This operation has its own name, one that has perhaps haunted our AQM group. That's right, it's the **Laplacian**, written $\nabla^2 \varphi$. 

In this context, it is measuring how much the value of $\varphi$ at a chosen point differs from the average of its neighbours. As such:

$$\nabla \cdot \mathbf{J} = -D \nabla^2 \varphi$$

Something important to note: 
- If $D$ were spatially varying (i.e., if the material were unhumbly inhomogeneous, and we do not use Assumption 3), we could  *not* pull it out of the divergence. 
- This would suck, as we'd be stuck with $\nabla \cdot (D \nabla \varphi)$ instead of $D \nabla^2 \varphi$. 
- And now you see why no slander towards Assumption 3 shall be accepted (beyond what's scientifically reasonable, of course).

### When an equation comes together...

Right, let's put this all together shall we? First off, let's substitute our result $\nabla \cdot \mathbf{J} = -D \nabla^2 \varphi$ right back into our neutron balance equation:

$$\frac{\partial n}{\partial t} = \nu \Sigma_f \varphi - \Sigma_a \varphi - \nabla \cdot \mathbf{J}$$

$$\frac{\partial n}{\partial t} = \nu \Sigma_f \varphi - \Sigma_a \varphi - (-D \nabla^2 \varphi)$$

The double negative resolve itself. By subtracting a negative divergence (this represents outward leakage of our neutrons, by the way), we get a positive Laplacian term:

$$\frac{\partial n}{\partial t} = D \nabla^2 \varphi + \nu \Sigma_f \varphi - \Sigma_a \varphi$$

Now, notice that everything on the right is in terms of $\varphi$, but the left side still has $n$. We want the whole equation in terms of $\varphi$. 

But aha!

Recall from earlier that $\varphi = nv$, where $v$ was the neutron speed. 

As such, we can a little bit of rearranging: 
$$
n = \varphi / v
$$ 

Since $v$ is constant (this is our one-group approximation! We are assuming all neutrons have the same energy. Imagine how impossible this would be if you took the entire scope of the fissile material's spectrum of neutron energies), we can substitute $n = \varphi / v$ into the time derivative:

$$\frac{\partial n}{\partial t} = \frac{\partial}{\partial t}\left(\frac{\varphi}{v}\right)$$

Since $v$ is a constant, let's pull it out of the derivative:

$$\frac{\partial n}{\partial t} = \frac{1}{v} \frac{\partial \varphi}{\partial t}$$

Now we substitute again:

$$\frac{1}{v} \frac{\partial \varphi}{\partial t} = D \nabla^2 \varphi + \nu \Sigma_f \varphi - \Sigma_a \varphi$$

Now, let's multiply both sides by $v$:

$$\frac{\partial \varphi}{\partial t} = vD \nabla^2 \varphi + v(\nu \Sigma_f - \Sigma_a) \varphi$$

And there we have it folks:

$$\boxed{\frac{\partial \varphi}{\partial t} = vD \nabla^2 \varphi + v(\nu \Sigma_f - \Sigma_a) \varphi}$$

This is the **one-group, time-dependent neutron diffusion equation**. Let's take a second to stare at it in all its glory, because every term is dripping with mathematical subtext:

- $\partial \varphi / \partial t$: this is how fast the neutron flux (neutron "traffic") is changing at a given point.

- $vD \nabla^2 \varphi$: this is the diffusion (leakage) term. The Laplacian $\nabla^2 \varphi$ measures just how much the flux at a point differs from the average of its neighbours. If the flux is locally higher than its surroundings ($\nabla^2 \varphi < 0$), neutrons diffuse outward and this term is negative: leakage *reduces* the flux. As you might have guessed, this is the term responsible for neutrons escaping through the surface!

- $v(\nu \Sigma_f - \Sigma_a) \varphi$: and this is the net production term. What drives the entire nuclear fission energy project! If $\nu \Sigma_f > \Sigma_a$ (fission production exceeds absorption), this term is positive and drives an exponential growth. If $\nu \Sigma_f < \Sigma_a$, it drives decay. For fissile materials like U-235 and Pu-239, $\nu \Sigma_f > \Sigma_a$, so the question of criticality is really about whether this net production can overcome the leakage losses. And this is something operators have to carefully control in a nuclear power station, to keep the reactor productive, but also safe.

Oh, and just one more thing to note: 
- For the steady-state criticality analysis we'll do later, we will set $\partial \varphi / \partial t = 0$ and thus, the factor $v$ drops out entirely. When we do time-dependent simulations later, we will keep it. 
- So, don't get confused when this term comes and goes.

---
## 4. Specialisation to Spherical Symmetry

So, where do we go from here? 

I guess the title gave it away. It's time to take into account the other core principle outside of one-group energy. Our bare (not bear, very important distinction) sphere. Before now, our lovely equation works for any shape: a cube, a cylinder, a blob, a EF76 Nebulon-B escort frigate, anything! The Laplacian $\nabla^2$ is general.

Let's specialise it to our lovely sphere. Now, the sphere is symmetric and the material is homogeneous (god bless Assumption 3), so there's no reason for the flux to depend on which direction you're looking from the centre. What it can only depend on is how far from the centre you are. 

So, $\varphi = \varphi(r, t)$, and thus the full 3D Laplacian collapses to *just* its radial component.

